To predict the probability of frog presence ($p$) for a specific site, we use the weights calculated by our Gradient Ascent algorithm. This process happens in two stages: calculating the Log-Odds ($z$) and passing them through the Sigmoid Function.

In [165]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [166]:
#Reading the CSV file
#df = pd.read_csv("../Data/frogs.csv", index_col=0)
df = pd.read_csv("data/frogs.csv", index_col=0)

In [167]:
#Adding the temperature columns for analysis
df["temp_diff"] = df["meanmax"] - df["meanmin"]
df["temp_mid"] = (df["meanmax"] + df["meanmin"])/2

#Adding the squares of some parameters to not be monotonous
temp_mean = df['temp_mid'].mean()
df['temp_mid_sq'] = (df['temp_mid']-temp_mean) ** 2

In [168]:
weights = np.array([0.0, 0.0, 0.0, 0.0])
learning_rates = [0.1, 0.5, 1.0, 2.0]
epoch_counts = [5000, 10000, 20000]
decay_rates = [0.0, 0.00001, 0.0001, 0.001] # 0.0 means no decay!

In [169]:
X_df = df[['distance', 'temp_mid', 'temp_mid_sq', 'altitude']]

X_df_with_intercept = sm.add_constant(X_df)

# Convert to a raw NumPy array for the math
X = X_df_with_intercept.to_numpy()

X_scaled = X.copy()
for i in range(1, X.shape[1]):
    col_mean = np.mean(X[:, i])
    col_std = np.std(X[:, i])
    X_scaled[:, i] = (X[:, i] - col_mean) / col_std

Y = df['pres.abs'].to_numpy()

In [170]:
def gradient(x, y, rate, n, decay_rate, weigh):
    m = len(y)
    for i in range(n):
       # Calculate the Log-Odds (z) and Probability (p)
       z = np.dot(x, weigh)
       p = 1 / (1 + np.exp(-z))

       # Calculate the Error (Actual Data - Predicted)
       error = y - p

       # The Gradient Equation
       # We multiply the features (X) by the error, and sum them up
       grad = np.dot(x.T, error) / m

       current_rate = rate * (1.0 / (1.0 + decay_rate * i))
       # Take a step up the hill by updating the weights
       weigh += current_rate * grad

       # Print the progress every 1000 steps so we aren't blind!
       if i % 1000 == 0:
            # Calculate the actual Log-Loss (Cross-Entropy) to watch it shrink
            loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
            print(f"Epoch {i} | Log-Loss: {loss:.4f}")
    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return weigh, loss

In [171]:
best_loss = float('inf')
best_lr = 0
best_ep = 0
best_weights = None

for lr in learning_rates:
    for ep in epoch_counts:
        for dr in decay_rates:
            # Reset weights to 0
            initial_weights = np.zeros(X_scaled.shape[1])

            final_weights, final_loss = gradient(X_scaled, Y, lr, ep, dr, initial_weights)

            print(f"LR: {lr} | Ep: {ep} | Decay: {dr} --> Loss: {final_loss:.4f}")

            if final_loss < best_loss:
                best_loss = final_loss
                best_loss = final_loss
                best_lr = lr
                best_ep = ep
                best_weights = final_weights
                best_decay = dr

print("-" * 30)
print(f"WINNER: Learning Rate {best_lr} with {best_ep} Epochs adn {best_decay} Decay!")
print(f"Lowest Error Achieved: {best_loss:.4f}")

Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.5234
Epoch 2000 | Log-Loss: 0.5220
Epoch 3000 | Log-Loss: 0.5210
Epoch 4000 | Log-Loss: 0.5200
LR: 0.1 | Ep: 5000 | Decay: 0.0 --> Loss: 0.5192
Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.5234
Epoch 2000 | Log-Loss: 0.5221
Epoch 3000 | Log-Loss: 0.5210
Epoch 4000 | Log-Loss: 0.5201
LR: 0.1 | Ep: 5000 | Decay: 1e-05 --> Loss: 0.5193
Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.5235
Epoch 2000 | Log-Loss: 0.5222
Epoch 3000 | Log-Loss: 0.5214
Epoch 4000 | Log-Loss: 0.5206
LR: 0.1 | Ep: 5000 | Decay: 0.0001 --> Loss: 0.5200
Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.5244
Epoch 2000 | Log-Loss: 0.5232
Epoch 3000 | Log-Loss: 0.5228
Epoch 4000 | Log-Loss: 0.5225
LR: 0.1 | Ep: 5000 | Decay: 0.001 --> Loss: 0.5223
Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.5234
Epoch 2000 | Log-Loss: 0.5220
Epoch 3000 | Log-Loss: 0.5210
Epoch 4000 | Log-Loss: 0.5200
Epoch 5000 | Log-Loss: 0.5192
Epoch 6000 | Log-Loss: 0.5184
Ep

In [172]:
means = np.mean(X_df.to_numpy(), axis=0)
stds = np.std(X_df.to_numpy(), axis=0)

scaled_intercept = best_weights[0]
scaled_feature_weights = best_weights[1:]

unscaled_feature_weights = scaled_feature_weights / stds

unscaled_intercept = scaled_intercept - np.sum((scaled_feature_weights * means) / stds)

unscaled_weights = np.insert(unscaled_feature_weights, 0, unscaled_intercept)

In [173]:
# Calculate the final probabilities using winning weights
final_z = np.dot(X_scaled, best_weights)
final_p = 1 / (1 + np.exp(-final_z))

# Draw a line at 50%: If probability is >= 0.5, predict 1 (Frog). Else 0.
predictions = (final_p >= 0.5).astype(int)

# Compare predictions to reality and calculate the percentage
accuracy = np.mean(predictions == Y)


print("-" * 30)
print(f"WINNER: Learning Rate {best_lr} with {best_ep} Epochs and {best_decay} Decay!")
print(f"Lowest Error Achieved: {best_loss:.4f}\n")

# Map the weights to their actual names
feature_names = ["Intercept", "Distance", "Temp_Mid", "Temp_Mid_Sq", "Altitude"]

print("Final Calculated Weights (normalised and not):")
for name, weight, unweight in zip(feature_names, best_weights, unscaled_weights):
    print(f"  {name}: {weight:.4f}, {unweight:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

------------------------------
WINNER: Learning Rate 2.0 with 20000 Epochs and 0.0 Decay!
Lowest Error Achieved: 0.5112

Final Calculated Weights (normalised and not):
  Intercept: -1.1732, -117.5187
  Distance: -1.6791, -0.0007
  Temp_Mid: 5.8348, 6.9139
  Temp_Mid_Sq: -0.5855, -0.6628
  Altitude: 4.8414, 0.0388
Accuracy: 77.83%


$$\Large p = \frac{1}{1 + e^{-(-117.5187 - 0.0007(\text{Distance}) + 6.9139(\text{Temp Mid}) - 0.6628(\text{Temp Mid} - \text{Mean})^2 + 0.0388(\text{Altitude}))}}$$


In [174]:
# Calculate the final probabilities (p) using the unscaled weights
z = np.dot(X, unscaled_weights)
p = 1 / (1 + np.exp(-z))

# Build the Diagonal Weight Matrix
# Formula: w_ii = p * (1 - p)
variances = p * (1 - p)
W = np.diag(variances)

# Calculate the Fisher Information Matrix (I)
# Formula: X^T * W * X
fisher_info = np.dot(np.dot(X.T, W), X)

# Calculate the Covariance Matrix (Sigma)
# Formula: Inverse of the Fisher Information Matrix
cov_matrix = np.linalg.inv(fisher_info)

# Extract the Standard Errors (SE)
# Formula: Square root of the diagonal elements of Sigma
standard_errors = np.sqrt(np.diag(cov_matrix))


def predict_and_summarize(x_new, weights, cov_mat, std_errs):
    # Define the z-critical values
    z_90, z_95, z_99 = 1.645, 1.960, 2.576

    # Probability Math (The Delta Method)
    z_val = np.dot(x_new, weights)

    # Calculate the variance of z
    var_z = np.dot(np.dot(x_new, cov_mat), x_new)
    se_z = np.sqrt(var_z)

    # Helper function for Sigmoid
    def sigmoid(val):
        return 1 / (1 + np.exp(-np.clip(val, -500, 500)))

    # Helper function to switch to scientific notation if < 0.001
    def format_prob(val):
        if val < 0.001:
            return f"{val:>8.2e}%"
        else:
            return f"{val:>8.2f}%"

    # Calculate Probability Estimate and its bounds (convert to percentages)
    p_est = sigmoid(z_val) * 100
    p_low_90, p_high_90 = sigmoid(z_val - z_90 * se_z) * 100, sigmoid(z_val + z_90 * se_z) * 100
    p_low_95, p_high_95 = sigmoid(z_val - z_95 * se_z) * 100, sigmoid(z_val + z_95 * se_z) * 100
    p_low_99, p_high_99 = sigmoid(z_val - z_99 * se_z) * 100, sigmoid(z_val + z_99 * se_z) * 100

    p_est_str = format_prob(p_est)
    p_90_str = f"[{format_prob(p_low_90)}, {format_prob(p_high_90)}]"
    p_95_str = f"[{format_prob(p_low_95)}, {format_prob(p_high_95)}]"
    p_99_str = f"[{format_prob(p_low_99)}, {format_prob(p_high_99)}]"

    # Print the Table Header
    print("-" * 98)
    print(f"{'Feature':<12} | {'Estimate':>10} | {'90% CI':>21} | {'95% CI':>21} | {'99% CI':>21}")
    print("-" * 98)

    print(f"{'Probability':<12} | {p_est_str:>10} | {p_90_str:>24} | {p_95_str:>24} | {p_99_str:>24}")
    print("-" * 98)
    print("-" * 98) # Separator to distinguish prediction from parameters

    # 5. Print the Feature Rows (Formatted as Raw Weights)
    feature_names = ["Intercept", "Distance", "Temp_Mid", "Temp_Mid_Sq", "Altitude"]

    for i in range(len(feature_names)):
        name = feature_names[i]
        w = weights[i]
        se = std_errs[i]

        # Calculate bounds
        low_90, high_90 = w - (z_90 * se), w + (z_90 * se)
        low_95, high_95 = w - (z_95 * se), w + (z_95 * se)
        low_99, high_99 = w - (z_99 * se), w + (z_99 * se)

        # Format strings
        ci_90_str = f"[{low_90:>9.4f}, {high_90:>8.4f}]"
        ci_95_str = f"[{low_95:>9.4f}, {high_95:>8.4f}]"
        ci_99_str = f"[{low_99:>9.4f}, {high_99:>8.4f}]"
        print(f"{name:<12} | {w:>10.4f} | {ci_90_str:>21} | {ci_95_str:>21} | {ci_99_str:>21}")
    print("-" * 98)

In [175]:
# [Intercept (always 1.0), Distance: 500m, Temp: 10 C, Altitude: 1500m]
distance_test = 500.0
temp_test = 10.0
altitude_test = 1500.0
new_site = np.array([1.0, distance_test, temp_test, (temp_test - temp_mean)**2, altitude_test])

print("\nPREDICTION FOR SITE: 500m Distance | 10°C Temp | 1500m Altitude")
predict_and_summarize(new_site, unscaled_weights, cov_matrix, standard_errors)


PREDICTION FOR SITE: 500m Distance | 10°C Temp | 1500m Altitude
--------------------------------------------------------------------------------------------------
Feature      |   Estimate |                90% CI |                95% CI |                99% CI
--------------------------------------------------------------------------------------------------
Probability  |     99.96% |   [   85.07%,   100.00%] |   [   64.12%,   100.00%] |   [   15.62%,   100.00%]
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Intercept    |  -117.5187 | [-204.6209, -30.4164] | [-221.3000, -13.7373] | [-253.9170,  18.8797]
Distance     |    -0.0007 | [  -0.0009,  -0.0004] | [  -0.0010,  -0.0003] | [  -0.0011,  -0.0002]
Temp_Mid     |     6.9139 | [   2.2770,  11.5508] | [   1.3891,  12.4387] | [  -0.3473,  14.1751]
Temp_Mid_Sq  |    -0.6628 | [  -1.1136, 

In [176]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

# This is the wrapper function that the sliders will talk to
def interactive_habitat_predictor(distance, temperature, altitude):
    centered_temp_sq = (temperature - temp_mean) ** 2
    new_site_binder = np.array([1.0, distance, temperature, centered_temp_sq, altitude])
    predict_and_summarize(new_site_binder, unscaled_weights, cov_matrix, standard_errors)

# Generate the UI!
widgets.interact(
    interactive_habitat_predictor,
    distance=widgets.FloatSlider(value=500, min=0, max=5000, step=10, description='Distance (m):'),
    temperature=widgets.FloatSlider(value=9.5, min=-10, max=40, step=0.5, description='Temp (°C):'),
    altitude=widgets.FloatSlider(value=1200, min=0, max=4000, step=50, description='Altitude (m):')
)

interactive(children=(FloatSlider(value=500.0, description='Distance (m):', max=5000.0, step=10.0), FloatSlide…

<function __main__.interactive_habitat_predictor(distance, temperature, altitude)>